## <center> Tune, Train, and Evaluate YOLO for Chestnut Burr Detection </center>

##### **Note:** This notebook uses relative paths. Set the working directory to the project root. 

In [ ]:
import os

# set to project root (up one level)
os.chdir("..")
print("Current working directory:", os.getcwd())

#### <center> Import libraries </center>

In [ ]:
from pathlib import Path
import torch

from burr_detection.utils import load_config
from burr_detection.dataset import prepare_dataset_splits, burr_tile_group_key
from burr_detection.training import YOLOTrainer
from burr_detection.tuning import YOLOTuner
from burr_detection.inference import YOLOInference

#### <center> Check for GPU (CUDA / MPS) </center>

In [ ]:
if torch.cuda.is_available():
    print(f"GPU (CUDA): {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.3f} GB")
elif torch.backends.mps.is_available():
    print("GPU (MPS): Apple Silicon")
else:
    print("No GPU (CUDA/MPS) available; training will run on CPU.")
    print("For NVIDIA GPUs, install CUDA PyTorch via 'pip3 install -U torch torchvision --index-url https://download.pytorch.org/whl/cu130'")

#### <center> Load config file </center>

In [ ]:
config = load_config("burr_detection/config.yml")

training_steps = config['training_steps']
training_params = config['training_params']
tuning_space = config['tuning_space']
inference_params = config['inference']
data_paths = config['data']

#### <center> Create train, val, test splits </center> 

In [ ]:
prepare_dataset_splits(images_dir=Path(data_paths['training_dir']) / 'images',
                       labels_dir=Path(data_paths['training_dir']) / 'labels',
                       output_dir=Path(data_paths['training_dir']),
                       splits=tuple(config.get('split', {}).get('fracs', [0.7, 0.2, 0.1])),
                       seed=config.get('split', {}).get('seed', 666),
                       group_key_fn=burr_tile_group_key)  # group by source tree -> no tile-level leakage

#### <center> Train YOLO </center>

In [ ]:
trainer = YOLOTrainer(model_size=str(training_params[0]['model_size']),
                      prints_per_epoch=5,
                      training_steps=training_steps) 

trainer.train(yolo_data_dir=data_paths['training_dir'],
              config=training_params[0],
              conf_threshold=inference_params['conf_threshold'],
              iou_threshold=inference_params['iou_threshold'],
              plot_mode='subset')

In [ ]:
print(f"Training complete! Results saved to: {trainer.output_dir}")
print(f"Training metrics CSV: {trainer.training_metrics_path}")
print(f"Best model weights: {trainer.final_weights_path}")
print(f"Best model path: {trainer.best_model_path}")
print(f"Test predictions: {trainer.test_preds}")
print(f"Metrics history (first 3): {trainer.metrics_history[:3]}")
print(f"Dataset YAML used: {trainer.yaml_path}")
print(f"Validation metrics (last epoch): {trainer.validation_metrics}")

#### <center> Tune YOLO </center>

In [ ]:
tuner = YOLOTuner(
    num_samples=50,
    max_concurrent_trials=2,
    yolo_data_dir=data_paths['training_dir'],
    training_steps=training_steps,
    points_to_evaluate=training_params,
    tuning_space=tuning_space,
    conf_threshold=inference_params['conf_threshold'],
    iou_threshold=inference_params['iou_threshold'],
    plot_mode='subset'
)

tuner.run()

In [ ]:
print(f"Tuning complete! Results saved to: {tuner.best_output_dir}")
print(f"Best tuned model: {tuner.best_model_path}")
print(f"Best trial path: {tuner.best_trial['path']}")
print(f"Best trial config: {tuner.best_trial['config']}")
print(f"Best trial metrics (first 3 rows):\n{tuner.best_trial['metrics_dataframe'].head(3)}")
print(f"Best trial model path: {tuner.best_trial['model_path']}")
print(f"Best trial output dir: {tuner.best_trial['output_dir']}")
print(f"Best trial predictions: {tuner.best_trial_preds}")
print(f"Best trial config dict: {tuner.best_trial_config}")
print(f"All tuning results object: {tuner.results}")

#### <center> YOLO Inference </center>

In [ ]:
inferencer = YOLOInference(
    model_path=inference_params['model_path'], # Default best model from last run (trainer.best_model_path or tuner.best_model_path too) 
    image_selections_path=data_paths['image_selections'],
    conf_threshold=inference_params['conf_threshold'],
    iou_threshold=inference_params['iou_threshold'],
    plot_mode='subset'
)
inferencer.run()

In [ ]:
print(f"Inference complete! Results saved to: {inferencer.output_dir}")
print(f"Detections CSV: {inferencer.csv_path}")
print(f"Results DataFrame (first 3 rows):\n{inferencer.results_df.head(3)}")
print(f"All predictions: {inferencer.all_predictions}")
print(f"Model path used: {inferencer.model_path}")